# 05 – Model Training

This notebook covers:
- Loading train / validation / test splits from `s3://ads508-team1-sephora/Model/final_splits/`
- Training two XGBoost models via SageMaker bring-your-own-script:
  - **Product Rating Regressor** → predicts numeric product rating
  - **Review Sentiment Classifier** → predicts high (≥4) vs low (<4) review rating
- Hyperparameter tuning with SageMaker Automatic Model Tuning (Bayesian)
- Evaluating both models on held-out test sets
- Saving evaluation results and trained model artifacts to S3

**Input:** `s3://ads508-team1-sephora/Model/final_splits/`  
**Output:** `s3://ads508-team1-sephora/Model/model_artifacts/`  
           `s3://ads508-team1-sephora/Model/evaluation/`


## 1. Imports & Configuration


In [9]:
import os
import json
import boto3
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import sagemaker
from sagemaker import get_execution_role
from sagemaker.xgboost import XGBoost
from sagemaker.tuner import (
    HyperparameterTuner,
    IntegerParameter,
    ContinuousParameter,
)

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix,
)

# ── AWS / SageMaker config ──────────────────────────────────────
sess        = sagemaker.Session()
region      = boto3.Session().region_name
role        = get_execution_role()

BUCKET            = "ads508-team1-sephora"
S3_SPLITS         = f"s3://{BUCKET}/Model/final_splits"
S3_TRAIN_INPUT    = f"s3://{BUCKET}/Model/train_input"
S3_MODEL_OUTPUT   = f"s3://{BUCKET}/Model/model_artifacts"
S3_EVAL_OUTPUT    = f"s3://{BUCKET}/Model/evaluation"
LOCAL_TMP         = "./temp_train"

os.makedirs(LOCAL_TMP, exist_ok=True)

print(f"Region  : {region}")
print(f"Role    : {role}")
print(f"Bucket  : {BUCKET}")

Region  : us-east-1
Role    : arn:aws:iam::471590717821:role/LabRole
Bucket  : ads508-team1-sephora


## 2. Load Train / Validation / Test Splits from S3


In [10]:
# ── Product splits ──────────────────────────────────────────────
X_train_prod = pd.read_parquet(f"{S3_SPLITS}/product_X_train.parquet")
X_val_prod   = pd.read_parquet(f"{S3_SPLITS}/product_X_val.parquet")
X_test_prod  = pd.read_parquet(f"{S3_SPLITS}/product_X_test.parquet")
y_train_prod = pd.read_parquet(f"{S3_SPLITS}/product_y_train.parquet")["rating"].values
y_val_prod   = pd.read_parquet(f"{S3_SPLITS}/product_y_val.parquet")["rating"].values
y_test_prod  = pd.read_parquet(f"{S3_SPLITS}/product_y_test.parquet")["rating"].values

# ── Review splits ───────────────────────────────────────────────
X_train_rev = pd.read_parquet(f"{S3_SPLITS}/review_X_train.parquet")
X_val_rev   = pd.read_parquet(f"{S3_SPLITS}/review_X_val.parquet")
X_test_rev  = pd.read_parquet(f"{S3_SPLITS}/review_X_test.parquet")
y_train_rev = pd.read_parquet(f"{S3_SPLITS}/review_y_train.parquet")["rating"].values
y_val_rev   = pd.read_parquet(f"{S3_SPLITS}/review_y_val.parquet")["rating"].values
y_test_rev  = pd.read_parquet(f"{S3_SPLITS}/review_y_test.parquet")["rating"].values

print("Product splits:")
print(f"  X_train : {X_train_prod.shape}  y_train : {y_train_prod.shape}")
print(f"  X_val   : {X_val_prod.shape}    y_val   : {y_val_prod.shape}")
print(f"  X_test  : {X_test_prod.shape}   y_test  : {y_test_prod.shape}")
print("\nReview splits:")
print(f"  X_train : {X_train_rev.shape}  y_train : {y_train_rev.shape}")
print(f"  X_val   : {X_val_rev.shape}    y_val   : {y_val_rev.shape}")
print(f"  X_test  : {X_test_rev.shape}   y_test  : {y_test_rev.shape}")

Product splits:
  X_train : (2132, 287)  y_train : (2132,)
  X_val   : (118, 287)    y_val   : (118,)
  X_test  : (119, 287)   y_test  : (119,)

Review splits:
  X_train : (9000, 3001)  y_train : (9000,)
  X_val   : (500, 3001)    y_val   : (500,)
  X_test  : (500, 3001)   y_test  : (500,)


## 3. Prepare Target Variables

- **Product model** → regression on numeric `rating`
- **Review model** → binary classification: high (rating ≥ 4) vs low (rating < 4)


In [11]:
# Binary label for review classifier
HIGH_RATING_THRESHOLD = 4

y_train_rev_bin = (y_train_rev >= HIGH_RATING_THRESHOLD).astype(int)
y_val_rev_bin   = (y_val_rev   >= HIGH_RATING_THRESHOLD).astype(int)
y_test_rev_bin  = (y_test_rev  >= HIGH_RATING_THRESHOLD).astype(int)

pos_rate = y_train_rev_bin.mean()
print(f"Review training set — high-rating rate : {pos_rate:.2%}")
print(f"Scale_pos_weight (neg/pos ratio)       : {(1 - pos_rate) / pos_rate:.2f}")

Review training set — high-rating rate : 81.78%
Scale_pos_weight (neg/pos ratio)       : 0.22


## 4. Upload Training Data to S3 for SageMaker

SageMaker XGBoost built-in expects CSV with the **label in the first column**.


In [12]:
s3_client = boto3.client("s3")

def upload_csv_to_s3(X, y, prefix, filename):
    """Concatenate label (first col) + features, save as CSV, upload to S3."""
    df = pd.concat(
        [pd.Series(y, name="label"), pd.DataFrame(X).reset_index(drop=True)],
        axis=1,
    )
    local_path = f"{LOCAL_TMP}/{filename}"
    s3_key     = f"{prefix}/{filename}"
    df.to_csv(local_path, index=False, header=False)
    s3_client.upload_file(local_path, BUCKET, s3_key)
    s3_uri = f"s3://{BUCKET}/{s3_key}"
    print(f"Uploaded  {s3_uri}  shape={df.shape}")
    return s3_uri

# Product — regression
upload_csv_to_s3(X_train_prod, y_train_prod, "Model/train_input/product", "train.csv")
upload_csv_to_s3(X_val_prod,   y_val_prod,   "Model/train_input/product", "validation.csv")

# Review — classification (binary labels)
upload_csv_to_s3(X_train_rev, y_train_rev_bin, "Model/train_input/review", "train.csv")
upload_csv_to_s3(X_val_rev,   y_val_rev_bin,   "Model/train_input/review", "validation.csv")

print("\nAll training CSVs uploaded.")

Uploaded  s3://ads508-team1-sephora/Model/train_input/product/train.csv  shape=(2132, 288)
Uploaded  s3://ads508-team1-sephora/Model/train_input/product/validation.csv  shape=(118, 288)
Uploaded  s3://ads508-team1-sephora/Model/train_input/review/train.csv  shape=(9000, 3002)
Uploaded  s3://ads508-team1-sephora/Model/train_input/review/validation.csv  shape=(500, 3002)

All training CSVs uploaded.


## 5. Model A – Product Rating Regressor (XGBoost)

**Objective:** `reg:squarederror`  
**Instance:** `ml.m5.xlarge` (4 vCPU / 16 GB) — sufficient for 2,132 rows × 287 features  
**Approach:** SageMaker built-in XGBoost container (bring-your-own-script not needed for tabular XGBoost)


In [13]:
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator

# Retrieve the SageMaker-managed XGBoost image URI
xgb_image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)
print(f"XGBoost image URI: {xgb_image_uri}")

# ── Estimator ──────────────────────────────────────────────────
xgb_product = Estimator(
    image_uri          = xgb_image_uri,
    role               = role,
    instance_count     = 1,
    instance_type      = "ml.m5.xlarge",
    output_path        = f"{S3_MODEL_OUTPUT}/product/",
    sagemaker_session  = sess,
    base_job_name      = "glowsense-product-xgb",
)

# ── Hyperparameters ────────────────────────────────────────────
xgb_product.set_hyperparameters(
    objective          = "reg:squarederror",
    num_round          = 300,          # boosting rounds; early stopping monitors val
    max_depth          = 5,            # controls tree depth / complexity
    eta                = 0.05,         # learning rate — low for better generalisation
    subsample          = 0.8,          # row sub-sampling per tree
    colsample_bytree   = 0.8,          # feature sub-sampling per tree
    min_child_weight   = 5,            # prevents splitting on very small groups
    early_stopping_rounds = 20,        # stop if val RMSE doesn't improve for 20 rounds
    eval_metric        = "rmse",
)

# ── Data Channels ──────────────────────────────────────────────
product_train_input = TrainingInput(
    s3_data      = f"s3://{BUCKET}/Model/train_input/product/train.csv",
    content_type = "text/csv",
)
product_val_input = TrainingInput(
    s3_data      = f"s3://{BUCKET}/Model/train_input/product/validation.csv",
    content_type = "text/csv",
)

print("Product estimator configured.")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


XGBoost image URI: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
Product estimator configured.


In [14]:
# ── Train ──────────────────────────────────────────────────────
xgb_product.fit(
    {"train": product_train_input, "validation": product_val_input},
    logs=True,
)

print("\nProduct model training complete.")
print(f"Model artifact : {xgb_product.model_data}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: glowsense-product-xgb-2026-03-31-02-02-54-904


2026-03-31 02:02:58 Starting - Starting the training job...
2026-03-31 02:03:12 Starting - Preparing the instances for training...
2026-03-31 02:04:00 Downloading - Downloading the training image......
2026-03-31 02:04:46 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-31 02:04:56.762 ip-10-0-224-157.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-31 02:04:56.825 ip-10-0-224-157.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-31:02:04:57:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-31:02:04:57:INFO] Failed to parse hyperparamete

## 6. Model B – Review Sentiment Classifier (XGBoost)

**Objective:** `binary:logistic`  
**Instance:** `ml.m5.2xlarge` (8 vCPU / 32 GB) — higher memory for 9,000 rows × 3,001 sparse TF-IDF features  
**Note:** `scale_pos_weight` corrects class imbalance (positivity bias in beauty reviews)


In [15]:
# Recompute scale_pos_weight
neg_count = (y_train_rev_bin == 0).sum()
pos_count = (y_train_rev_bin == 1).sum()
scale_pos_weight = round(neg_count / pos_count, 4)
print(f"neg_count={neg_count}  pos_count={pos_count}  scale_pos_weight={scale_pos_weight}")

# ── Estimator ──────────────────────────────────────────────────
xgb_review = Estimator(
    image_uri          = xgb_image_uri,
    role               = role,
    instance_count     = 1,
    instance_type      = "ml.m5.2xlarge",   # higher memory for large TF-IDF matrix
    output_path        = f"{S3_MODEL_OUTPUT}/review/",
    sagemaker_session  = sess,
    base_job_name      = "glowsense-review-xgb",
)

# ── Hyperparameters ────────────────────────────────────────────
xgb_review.set_hyperparameters(
    objective             = "binary:logistic",
    num_round             = 300,
    max_depth             = 4,              # shallower trees for high-dim sparse input
    eta                   = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.3,            # sample fewer features — important for TF-IDF
    min_child_weight      = 10,
    scale_pos_weight      = scale_pos_weight,
    early_stopping_rounds = 20,
    eval_metric           = "auc",
)

# ── Data Channels ──────────────────────────────────────────────
review_train_input = TrainingInput(
    s3_data      = f"s3://{BUCKET}/Model/train_input/review/train.csv",
    content_type = "text/csv",
)
review_val_input = TrainingInput(
    s3_data      = f"s3://{BUCKET}/Model/train_input/review/validation.csv",
    content_type = "text/csv",
)

print("Review estimator configured.")

neg_count=1640  pos_count=7360  scale_pos_weight=0.2228
Review estimator configured.


In [16]:
# ── Train ──────────────────────────────────────────────────────
xgb_review.fit(
    {"train": review_train_input, "validation": review_val_input},
    logs=True,
)

print("\nReview model training complete.")
print(f"Model artifact : {xgb_review.model_data}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: glowsense-review-xgb-2026-03-31-02-05-42-188
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # ── Train ──────────────────────────────────────────────────────                            │
│ ❱ 2 xgb_review.fit(                                                                              │
│   3 │   {"train": review_train_input, "validation": review_val_input},                           │
│   4 │   logs=True,                                                                               │
│   5 )                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:167 in wrapper  │
│                                                                                                  │
│   164 │   │   │   │   │   caught_ex = e                                                          │
│   165 │   │   │   │   finally:                                                                   │
│   166 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 167 │   │   │   │   │   │   raise caught_ex                                                    │
│   168 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   169 │   │   │   else:                                                                          │
│   170 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:138 in wrapper  │
│                                                                                                  │
│   135 │   │   │   │   start_timer = perf_counter()                                               │
│   136 │   │   │   │   try:                                                                       │
│   137 │   │   │   │   │   # Call the original function                                           │
│ ❱ 138 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   139 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   140 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   141 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/estimator

## 7. (Optional) Hyperparameter Tuning – Product Model

SageMaker Automatic Model Tuning with **Bayesian search** to refine `eta`, `max_depth`, and `subsample`.  
Uncomment and run this cell if you want to launch an HPO job (adds ~30–60 min).


In [ ]:
# ─── OPTIONAL: Hyperparameter Optimisation ────────────────────
# Uncomment to run HPO (recommended before final evaluation)

# hyperparameter_ranges = {
#     "eta":              ContinuousParameter(0.01, 0.2),
#     "max_depth":        IntegerParameter(3, 7),
#     "subsample":        ContinuousParameter(0.6, 1.0),
#     "colsample_bytree": ContinuousParameter(0.5, 1.0),
#     "min_child_weight": IntegerParameter(1, 10),
# }

# tuner_product = HyperparameterTuner(
#     estimator              = xgb_product,
#     objective_metric_name  = "validation:rmse",
#     objective_type         = "Minimize",
#     hyperparameter_ranges  = hyperparameter_ranges,
#     max_jobs               = 12,
#     max_parallel_jobs      = 3,
#     strategy               = "Bayesian",
#     base_tuning_job_name   = "glowsense-product-hpo",
# )

# tuner_product.fit(
#     {"train": product_train_input, "validation": product_val_input},
#     include_cls_metadata=False,
# )

# print("HPO job submitted — monitor in SageMaker console.")

print("HPO block is commented out. Uncomment to run tuning.")

## 8. Deploy Models for Batch Inference

We use **batch transform** instead of a persistent endpoint to keep costs low  
for a 3-person startup environment.


In [ ]:
# Save test sets to S3 for batch transform input
def upload_test_csv(X, prefix, filename):
    df         = pd.DataFrame(X).reset_index(drop=True)
    local_path = f"{LOCAL_TMP}/{filename}"
    s3_key     = f"{prefix}/{filename}"
    df.to_csv(local_path, index=False, header=False)
    s3_client.upload_file(local_path, BUCKET, s3_key)
    s3_uri = f"s3://{BUCKET}/{s3_key}"
    print(f"Uploaded test input : {s3_uri}")
    return s3_uri

prod_test_uri   = upload_test_csv(X_test_prod, "Model/batch_input/product", "test.csv")
review_test_uri = upload_test_csv(X_test_rev,  "Model/batch_input/review",  "test.csv")

In [ ]:
# ── Product batch transform ────────────────────────────────────
product_transformer = xgb_product.transformer(
    instance_count  = 1,
    instance_type   = "ml.m5.xlarge",
    output_path     = f"s3://{BUCKET}/Model/batch_output/product/",
    accept          = "text/csv",
    assemble_with   = "Line",
)
product_transformer.transform(
    data             = prod_test_uri,
    content_type     = "text/csv",
    split_type       = "Line",
    wait             = True,
    logs             = True,
)
print("Product batch transform complete.")

In [ ]:
# ── Review batch transform ─────────────────────────────────────
review_transformer = xgb_review.transformer(
    instance_count  = 1,
    instance_type   = "ml.m5.2xlarge",
    output_path     = f"s3://{BUCKET}/Model/batch_output/review/",
    accept          = "text/csv",
    assemble_with   = "Line",
)
review_transformer.transform(
    data             = review_test_uri,
    content_type     = "text/csv",
    split_type       = "Line",
    wait             = True,
    logs             = True,
)
print("Review batch transform complete.")

## 9. Load Predictions & Evaluate


In [ ]:
def load_batch_predictions(s3_output_prefix, output_filename="test.csv.out"):
    """Download batch transform output from S3 and return as numpy array."""
    local_out = f"{LOCAL_TMP}/{output_filename}"
    s3_key    = f"{s3_output_prefix}/{output_filename}".replace(f"s3://{BUCKET}/", "")
    s3_client.download_file(BUCKET, s3_key, local_out)
    preds = pd.read_csv(local_out, header=None).squeeze().values
    return preds

y_pred_prod   = load_batch_predictions("Model/batch_output/product")
y_pred_review = load_batch_predictions("Model/batch_output/review")

print(f"Product predictions  shape : {y_pred_prod.shape}")
print(f"Review predictions   shape : {y_pred_review.shape}")

### 9.1 Product Rating Regressor – Evaluation


In [ ]:
rmse  = np.sqrt(mean_squared_error(y_test_prod, y_pred_prod))
mae   = mean_absolute_error(y_test_prod, y_pred_prod)
r2    = r2_score(y_test_prod, y_pred_prod)

print("=" * 40)
print("Product Rating Regressor – Test Set")
print("=" * 40)
print(f"  RMSE : {rmse:.4f}")
print(f"  MAE  : {mae:.4f}")
print(f"  R²   : {r2:.4f}")

# ── Actual vs Predicted scatter ────────────────────────────────
plt.figure(figsize=(7, 5))
plt.scatter(y_test_prod, y_pred_prod, alpha=0.5, edgecolors="k", linewidths=0.3)
lims = [min(y_test_prod.min(), y_pred_prod.min()),
        max(y_test_prod.max(), y_pred_prod.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title("Product Model — Actual vs. Predicted")
plt.legend()
plt.tight_layout()
plt.savefig(f"{LOCAL_TMP}/product_actual_vs_pred.png", dpi=150)
plt.show()

### 9.2 Review Sentiment Classifier – Evaluation


In [ ]:
# XGBoost binary:logistic outputs probabilities → threshold at 0.5
y_pred_rev_bin = (y_pred_review >= 0.5).astype(int)

acc       = accuracy_score(y_test_rev_bin,  y_pred_rev_bin)
precision = precision_score(y_test_rev_bin, y_pred_rev_bin, zero_division=0)
recall    = recall_score(y_test_rev_bin,    y_pred_rev_bin, zero_division=0)
f1        = f1_score(y_test_rev_bin,        y_pred_rev_bin, zero_division=0)
roc_auc   = roc_auc_score(y_test_rev_bin,   y_pred_review)

print("=" * 40)
print("Review Sentiment Classifier – Test Set")
print("=" * 40)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print()
print(classification_report(y_test_rev_bin, y_pred_rev_bin,
                             target_names=["Low Rating", "High Rating"]))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────
cm = confusion_matrix(y_test_rev_bin, y_pred_rev_bin)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Pred Low", "Pred High"],
    yticklabels=["Actual Low", "Actual High"],
)
plt.title("Review Classifier — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{LOCAL_TMP}/review_confusion_matrix.png", dpi=150)
plt.show()

## 10. Feature Importance


In [ ]:
import tarfile
import xgboost as xgb

def load_xgb_model_from_s3(model_data_uri, local_name):
    """Download model.tar.gz from S3, extract, and load XGBoost Booster."""
    tar_path = f"{LOCAL_TMP}/{local_name}.tar.gz"
    model_dir = f"{LOCAL_TMP}/{local_name}_model"
    os.makedirs(model_dir, exist_ok=True)

    # Parse bucket / key from s3 URI
    s3_parts = model_data_uri.replace("s3://", "").split("/", 1)
    s3_client.download_file(s3_parts[0], s3_parts[1], tar_path)

    with tarfile.open(tar_path) as tar:
        tar.extractall(model_dir)

    model_file = next(
        os.path.join(model_dir, f)
        for f in os.listdir(model_dir)
        if f.endswith(".model") or f == "xgboost-model"
    )
    booster = xgb.Booster()
    booster.load_model(model_file)
    return booster

booster_prod   = load_xgb_model_from_s3(xgb_product.model_data, "product")
booster_review = load_xgb_model_from_s3(xgb_review.model_data,  "review")
print("Boosters loaded.")

In [ ]:
def plot_feature_importance(booster, feature_names, title, top_n=20):
    importance = booster.get_score(importance_type="gain")
    # Map f0, f1 … to actual column names if available
    if feature_names is not None:
        importance = {
            feature_names[int(k[1:])] if k.startswith("f") else k: v
            for k, v in importance.items()
            if k.startswith("f") and int(k[1:]) < len(feature_names)
        }
    imp_df = (
        pd.Series(importance, name="gain")
        .sort_values(ascending=False)
        .head(top_n)
        .reset_index()
    )
    imp_df.columns = ["Feature", "Gain"]

    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df, x="Gain", y="Feature", palette="viridis")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{LOCAL_TMP}/{title.replace(' ', '_').lower()}.png", dpi=150)
    plt.show()
    return imp_df

prod_feat_names   = list(X_train_prod.columns)
review_feat_names = list(X_train_rev.columns)

prod_importance   = plot_feature_importance(
    booster_prod,   prod_feat_names,   "Product Model — Top 20 Features by Gain"
)
review_importance = plot_feature_importance(
    booster_review, review_feat_names, "Review Model — Top 20 Features by Gain"
)

## 11. Save Evaluation Results to S3


In [ ]:
eval_results = {
    "product_regressor": {
        "rmse":  round(float(rmse), 4),
        "mae":   round(float(mae),  4),
        "r2":    round(float(r2),   4),
    },
    "review_classifier": {
        "accuracy":  round(float(acc),       4),
        "precision": round(float(precision), 4),
        "recall":    round(float(recall),    4),
        "f1_score":  round(float(f1),        4),
        "roc_auc":   round(float(roc_auc),   4),
    },
}

# Save JSON
eval_path = f"{LOCAL_TMP}/evaluation_results.json"
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)

s3_client.upload_file(eval_path, BUCKET, "Model/evaluation/evaluation_results.json")

# Upload plots
for img in ["product_actual_vs_pred.png", "review_confusion_matrix.png",
            "product_model_—_top_20_features_by_gain.png",
            "review_model_—_top_20_features_by_gain.png"]:
    local_img = f"{LOCAL_TMP}/{img}"
    if os.path.exists(local_img):
        s3_client.upload_file(local_img, BUCKET, f"Model/evaluation/{img}")
        print(f"Uploaded {img}")

print("\nEvaluation results saved:")
print(json.dumps(eval_results, indent=2))

In [ ]:
# Cleanup local temp files
shutil.rmtree(LOCAL_TMP)
print("Temp directory cleaned up.")

## Summary

| Step | Model | Details |
|------|-------|---------|
| Input | Both | `Model/final_splits/*.parquet` |
| Training | Product Regressor | XGBoost `reg:squarederror`, `ml.m5.xlarge`, 287 features |
| Training | Review Classifier | XGBoost `binary:logistic`, `ml.m5.2xlarge`, 3001 TF-IDF features |
| Tuning | Optional HPO | SageMaker Bayesian search, 12 jobs |
| Inference | Batch Transform | No persistent endpoint → cost-efficient |
| Evaluation | Product | RMSE, MAE, R² |
| Evaluation | Review | Accuracy, Precision, Recall, F1, ROC-AUC |
| Output | Artifacts | `Model/model_artifacts/` |
| Output | Evaluation | `Model/evaluation/evaluation_results.json` + plots |

➡ Next notebook: `06_future_enhancements.ipynb`
